debug a model 

1. Shape mismatch
2. Learning rate problem
3. Normalization problem
4. Wrong loss function
5. General training/debugging issues

In [53]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [54]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [55]:
dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [56]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

In [57]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [58]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Label shape:", labels.shape)

Image shape: torch.Size([64, 1, 28, 28])
Label shape: torch.Size([64])


In [59]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(28 * 28, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [60]:
model = MLP()

print(model)

MLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [61]:
outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 1, 28, 28])
Output shape: torch.Size([64, 10])


In [62]:
nn.Linear(28 * 28, 256)

Linear(in_features=784, out_features=256, bias=True)

In [63]:
nn.Linear(28 * 28, 128)

Linear(in_features=784, out_features=128, bias=True)

In [64]:
outputs = model(images)

In [65]:
nn.Linear(28 * 28, 256)

Linear(in_features=784, out_features=256, bias=True)

1st rule of debugging 

In [66]:
print(model)

MLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [67]:
print("Input:", images.shape)
print("Output:", model(images).shape)

Input: torch.Size([64, 1, 28, 28])
Output: torch.Size([64, 10])


set up loss function 

In [68]:
criterion = nn.CrossEntropyLoss()

optimizer 

In [69]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [70]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_loss:.4f}"
    )

Epoch [1/5], Loss: 0.2547
Epoch [2/5], Loss: 0.1015
Epoch [3/5], Loss: 0.0696
Epoch [4/5], Loss: 0.0514
Epoch [5/5], Loss: 0.0430


In [71]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [72]:
lr=1

In [73]:
optimizer = optim.Adam(
    model.parameters(),
    lr=1
)

In [74]:
lr=0.001

In [75]:
lr=0.0000001

Epoch 1 Loss: 2.31
Epoch 2 Loss: 2.30
Epoch 3 Loss: 2.30
Epoch 4 Loss: 2.29

In [76]:
transforms.Normalize(
    (0.1307,),
    (0.3081,)
)

Normalize(mean=(0.1307,), std=(0.3081,))

In [77]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [78]:
transforms.Normalize(
    (0.1307,),
    (0.3081,)
)

Normalize(mean=(0.1307,), std=(0.3081,))

wrong loss function 


In [79]:
criterion = nn.CrossEntropyLoss()

In [80]:
criterion = nn.MSELoss()

In [81]:
outputs = model(images)

print(outputs.shape)

torch.Size([64, 10])


In [82]:
predictions = outputs.argmax(dim=1)

In [83]:
print(predictions[:10])
print(labels[:10])

tensor([3, 1, 8, 9, 2, 6, 3, 7, 7, 0])
tensor([3, 1, 8, 9, 2, 6, 3, 7, 7, 0])


forgetting optimizer 

1. Check dataset
        ↓
2. Check input shape
        ↓
3. Check output shape
        ↓
4. Check labels
        ↓
5. Check loss function
        ↓
6. Check learning rate
        ↓
7. Check gradients
        ↓
8. Check optimizer.step()
        ↓
9. Check normalization
        ↓
10. Try overfitting a tiny batch

| Cause                       | Symptom                         | Fix                                         |
| --------------------------- | ------------------------------- | ------------------------------------------- |
| Shape mismatch              | RuntimeError                    | Check tensor dimensions                     |
| Wrong learning rate         | Loss explodes or barely changes | Tune LR                                     |
| Bad normalization           | Slow/unstable training          | Scale/normalize inputs                      |
| Wrong loss function         | Model learns poorly             | Match loss to task                          |
| Optimizer/training-loop bug | Weights don't update            | Check `zero_grad()`, `backward()`, `step()` |


WHY_MODEL_WAS_NOT_TRAINING.md

# Why My Model Wasn't Training

## Objective

The objective was to debug a PyTorch model that was not learning
and identify common causes of training failure.

---

## Cause 1: Shape Mismatch

### Problem

The input image had shape:

[64, 1, 28, 28]

After flattening, each image contains:

28 × 28 = 784 features.

The Linear layer must therefore accept 784 input features.

### Wrong

```python
nn.Linear(28 * 27, 256)